In [1]:
import sys
import os
import time
import gc

sys.path.append("..")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

from common import *

torch.set_num_threads(12)

bench = {
    lang: load_benchmark(lang)
    for lang in LANGS
}

print("Languages:", LANGS)
print("Number of languages:", len(LANGS))

Languages: ['as', 'bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'ne', 'or', 'pa', 'ta', 'te', 'ur']
Number of languages: 13


In [2]:
VARIANTS = {
    "mE5-small": "intfloat/multilingual-e5-small",
    "mE5-base": "intfloat/multilingual-e5-base",
    "mE5-large": "intfloat/multilingual-e5-large",
}

print(VARIANTS)

{'mE5-small': 'intfloat/multilingual-e5-small', 'mE5-base': 'intfloat/multilingual-e5-base', 'mE5-large': 'intfloat/multilingual-e5-large'}


In [3]:
model = SentenceTransformer(
    VARIANTS["mE5-small"],
    device="cpu"
)

model.max_seq_length = 512

print(model)
print("Max sequence length:", model.max_seq_length)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
Max sequence length: 512


In [4]:
query_prefix = "query: "
passage_prefix = "passage: "

print("Query example:")
print(query_prefix + bench["hi"][0]["query"].iloc[0])

print("\nPassage example:")
print(passage_prefix + bench["hi"][1]["text"].iloc[0])

Query example:
query: हिरलूम टमाटर का क्या अर्थ है

Passage example:
passage: डलदल की पूर्ण परिभाषा 1 एक आर्द्रभूमि जो अक्सर आंशिक या अस्थायी रूप से पानी से ढकी होती है, विशेष रूप से एक जो लकड़ी की वनस्पतियों से प्रभुत्व होती है 2 डलदल का एक टुकड़ा 3 एक कठिन या परेशान करने वाली स्थिति या विषय डलदल विशेषण


In [5]:
lang = "hi"

q, c, gold = bench[lang]

print("Queries:", len(q))
print("Documents:", len(c))
print("Gold labels:", len(gold))

Queries: 1000
Documents: 1000
Gold labels: 1000


In [6]:
t0 = time.time()

Eq = encode_cached(
    model,
    q["query"].tolist(),
    EMB_DIR / "mE5-small_hi_q.npy",
    prefix="query: ",
    batch_size=32,
)

print(
    "Query embeddings:",
    Eq.shape
)

print(
    f"Query encoding time: {(time.time() - t0):.1f}s"
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Query embeddings: (1000, 384)
Query encoding time: 5.9s


In [7]:
t0 = time.time()

Ed = encode_cached(
    model,
    c["text"].tolist(),
    EMB_DIR / "mE5-small_hi_d.npy",
    prefix="passage: ",
    batch_size=16,
)

print(
    "Document embeddings:",
    Ed.shape
)

print(
    f"Document encoding time: {(time.time() - t0):.1f}s"
)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Document embeddings: (1000, 384)
Document encoding time: 37.6s


In [8]:
S = Eq @ Ed.T

print("Score matrix:", S.shape)

Score matrix: (1000, 1000)


In [9]:
metrics = record(
    "mE5-small",
    "hi",
    S,
    gold,
)

print("\nE5-small Hindi")
print("=" * 40)

for key, value in metrics.items():
    print(f"{key}: {value}")


E5-small Hindi
MRR: 0.8787864090473311
MRR@10: 0.877061507936508
R@1: 0.831
R@5: 0.936
R@10: 0.951
R@20: 0.964
R@100: 0.989
nDCG@10: 0.8953804165862594
n_queries: 1000
model: mE5-small
key: hi
setting: mono
n_docs: 1000


In [10]:
print(
    "First query norm:",
    np.linalg.norm(Eq[0])
)

print(
    "First document norm:",
    np.linalg.norm(Ed[0])
)

First query norm: 0.99999994
First document norm: 0.99999994


In [11]:
def run_e5_variant(
    tag,
    hf_id,
    langs=LANGS,
):
    print("=" * 70)
    print(f"Running {tag}")
    print(f"Model: {hf_id}")
    print("=" * 70)

    model = SentenceTransformer(
        hf_id,
        device="cpu"
    )

    model.max_seq_length = 512

    for lang in langs:

        t0 = time.time()

        q, c, gold = bench[lang]

        Eq = encode_cached(
            model,
            q["query"].tolist(),
            EMB_DIR / f"{tag}_{lang}_q.npy",
            prefix="query: ",
            batch_size=32,
        )

        Ed = encode_cached(
            model,
            c["text"].tolist(),
            EMB_DIR / f"{tag}_{lang}_d.npy",
            prefix="passage: ",
            batch_size=16,
        )

        S = Eq @ Ed.T

        metrics = record(
            tag,
            lang,
            S,
            gold,
        )

        elapsed = (
            time.time() - t0
        ) / 60

        print(
            f"{lang} | "
            f"MRR={metrics['MRR']:.4f} | "
            f"MRR@10={metrics['MRR@10']:.4f} | "
            f"R@1={metrics['R@1']:.4f} | "
            f"R@10={metrics['R@10']:.4f} | "
            f"R@100={metrics['R@100']:.4f} | "
            f"{elapsed:.1f} min"
        )

    del model
    gc.collect()

In [12]:
run_e5_variant(
    "mE5-small",
    VARIANTS["mE5-small"],
)

Running mE5-small
Model: intfloat/multilingual-e5-small


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

as | MRR=0.6324 | MRR@10=0.6256 | R@1=0.5420 | R@10=0.8020 | R@100=0.9490 | 2.4 min


...

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

te | MRR=0.7767 | MRR@10=0.7737 | R@1=0.7070 | R@10=0.8980 | R@100=0.9670 | 2.5 min


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

ur | MRR=0.7515 | MRR@10=0.7475 | R@1=0.6750 | R@10=0.8860 | R@100=0.9730 | 1.5 min


In [13]:
run_e5_variant(
    "mE5-base",
    VARIANTS["mE5-base"],
)

Running mE5-base
Model: intfloat/multilingual-e5-base


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

te | MRR=0.8021 | MRR@10=0.7993 | R@1=0.7390 | R@10=0.9040 | R@100=0.9740 | 3.6 min


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

ur | MRR=0.7788 | MRR@10=0.7753 | R@1=0.7140 | R@10=0.8960 | R@100=0.9740 | 1.9 min


In [14]:
run_e5_variant(
    "mE5-large",
    VARIANTS["mE5-large"],
)

Running mE5-large
Model: intfloat/multilingual-e5-large


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

te | MRR=0.8576 | MRR@10=0.8557 | R@1=0.8090 | R@10=0.9390 | R@100=0.9820 | 18.1 min


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

ur | MRR=0.8380 | MRR@10=0.8360 | R@1=0.7780 | R@10=0.9310 | R@100=0.9760 | 16.7 min


In [ ]:
import pandas as pd
from pathlib import Path

metrics_path = Path("../results/metrics.csv")

metrics_df = pd.read_csv(metrics_path)

print(metrics_df.shape)
print(metrics_df.columns.tolist())

(52, 13)
['MRR', 'MRR@10', 'R@1', 'R@5', 'R@10', 'R@20', 'R@100', 'nDCG@10', 'n_queries', 'model', 'key', 'setting', 'n_docs']


In [17]:
e5_results = metrics_df[
    metrics_df["model"].isin(
        ["mE5-small", "mE5-base", "mE5-large"]
    )
].copy()

print(
    e5_results[
        [
            "model",
            "key",
            "MRR",
            "R@1",
            "R@5",
            "R@10",
            "R@20",
            "R@100",
        ]
    ].to_string(index=False)
)

    model key      MRR      R@1      R@5     R@10     R@20    R@100
mE5-small  as 0.632414 0.542000 0.751000 0.802000 0.858000 0.949000
mE5-small  bn 0.777265 0.704000 0.858000 0.903000 0.939000 0.978000
mE5-small  gu 0.690013 0.607608 0.789790 0.843844 0.894895 0.961962
mE5-small  hi 0.878786 0.831000 0.936000 0.951000 0.964000 0.989000
mE5-small  kn 0.761623 0.692000 0.849000 0.875000 0.913000 0.963000
mE5-small  ml 0.754775 0.673000 0.849000 0.892000 0.923000 0.978000
mE5-small  mr 0.772646 0.706000 0.854000 0.898000 0.925000 0.966000
mE5-small  ne 0.785844 0.717000 0.865000 0.906000 0.934000 0.971000
mE5-small  or 0.682721 0.602000 0.774000 0.834000 0.884000 0.949000
mE5-small  pa 0.694741 0.615000 0.787000 0.835000 0.882000 0.957000
mE5-small  ta 0.750147 0.688000 0.820000 0.865000 0.900000 0.962000
mE5-small  te 0.776685 0.707000 0.860000 0.898000 0.923000 0.967000
mE5-small  ur 0.751466 0.675000 0.841000 0.886000 0.922000 0.973000
 mE5-base  as 0.702802 0.620000 0.797000 0.85100

In [18]:
import pandas as pd
from pathlib import Path

metrics_df = pd.read_csv("../results/metrics.csv")

comparison = (
    metrics_df[
        metrics_df["model"].isin(
            [
                "bm25",
                "mE5-small",
                "mE5-base",
                "mE5-large",
            ]
        )
    ]
    .pivot(
        index="key",
        columns="model",
        values="MRR",
    )
)

comparison = comparison[
    [
        "bm25",
        "mE5-small",
        "mE5-base",
        "mE5-large",
    ]
]

comparison.columns = [
    "BM25",
    "E5-small",
    "E5-base",
    "E5-large",
]

print(comparison.to_string(
    float_format=lambda x: f"{x:.4f}"
))

      BM25  E5-small  E5-base  E5-large
key                                    
as  0.4616    0.6324   0.7028    0.7706
bn  0.6189    0.7773   0.8089    0.8709
gu  0.5465    0.6900   0.7284    0.8035
hi  0.7117    0.8788   0.8764    0.9101
kn  0.5380    0.7616   0.7717    0.8193
ml  0.5248    0.7548   0.7747    0.8406
mr  0.5097    0.7726   0.8001    0.8539
ne  0.5052    0.7858   0.7900    0.8446
or  0.5172    0.6827   0.7085    0.7815
pa  0.6532    0.6947   0.7358    0.8332
ta  0.5255    0.7501   0.7742    0.8226
te  0.5814    0.7767   0.8021    0.8576
ur  0.6529    0.7515   0.7788    0.8380


In [20]:
final_mrr = comparison.join(
    bge_df,
    how="inner"
)

final_mrr.index.name = "language"

print(
    final_mrr.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

           BM25  E5-small  E5-base  E5-large  BGE-M3
language                                            
as       0.4616    0.6324   0.7028    0.7706  0.8073
bn       0.6189    0.7773   0.8089    0.8709  0.8602
gu       0.5465    0.6900   0.7284    0.8035  0.8336
hi       0.7117    0.8788   0.8764    0.9101  0.8863
kn       0.5380    0.7616   0.7717    0.8193  0.8379
ml       0.5248    0.7548   0.7747    0.8406  0.8405
mr       0.5097    0.7726   0.8001    0.8539  0.8521
ne       0.5052    0.7858   0.7900    0.8446  0.8504
or       0.5172    0.6827   0.7085    0.7815  0.7953
pa       0.6532    0.6947   0.7358    0.8332  0.8298
ta       0.5255    0.7501   0.7742    0.8226  0.8078
te       0.5814    0.7767   0.8021    0.8576  0.8611
ur       0.6529    0.7515   0.7788    0.8380  0.8365


In [21]:
print(
    final_mrr.mean()
    .sort_values(
        ascending=False
    )
    .to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

BGE-M3     0.8384
E5-large   0.8343
E5-base    0.7733
E5-small   0.7469
BM25       0.5651


In [22]:
e5_means = (
    e5_results
    .groupby("model")["MRR"]
    .mean()
)

print(
    e5_means.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

model
mE5-base    0.7733
mE5-large   0.8343
mE5-small   0.7469


In [23]:
final_mrr.to_csv(
    "../results/retriever_mrr_comparison.csv"
)

print(
    "Saved: ../results/retriever_mrr_comparison.csv"
)

Saved: ../results/retriever_mrr_comparison.csv
